In [0]:
# Install required library to read Excel files
%pip install openpyxl

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# AUDIENCE PROFILING
# Using df_memb for customer level analysis and rfm table which already has segments
# -----------------------------------------------------------------------------------

# Load cleaned member data
df_memb = pd.read_parquet("/Volumes/workspace/default/retail_data/df_memb.parquet")

# Load RFM table with segments already assigned
rfm = pd.read_parquet("/Volumes/workspace/default/retail_data/rfm.parquet")

# Merge transaction data with segments
# This gives us every transaction labelled with its customer's segment
df_profiling = df_memb.merge(rfm[['Customer ID', 'Segment']], on='Customer ID', how='left')

print(f"Data loaded successfully")
print(f"\nMember transactions: {len(df_memb):,}")
print(f"Customers with segments: {len(rfm):,}")
print(f"Profiling dataset: {len(df_profiling):,} rows")
print(f"\nSegments in dataset:")
print(df_profiling['Segment'].value_counts())

print(f"\nLoad top 10 rows")
print(df_profiling.head(10).to_string(index=False))

In [0]:
# GEOGRAPHIC PROFILING BY SEGMENT
# Which countries do each segment come from?
# Helps identify where to focus retention and acquisition efforts geographically
# -------------------------------------------------------------------------------

# Get unique customer per segment and country
# We use rfm merged with df_memb to get country per customer
customer_country = df_memb.groupby('Customer ID')['Country'].first().reset_index()

# Merge with RFM to get segment + country per customer
segment_country = rfm[['Customer ID', 'Segment']].merge(customer_country, on='Customer ID')

# Count customers per segment per country
geo_profile = segment_country.groupby(['Segment', 'Country'])['Customer ID'].count().reset_index()
geo_profile.columns = ['Segment', 'Country', 'Customer Count']

# Focus on top 5 countries per segment
top_countries_per_segment = geo_profile.sort_values(
    ['Segment', 'Customer Count'], ascending=[True, False]
).groupby('Segment').head(5)

# Plot
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

segments = rfm['Segment'].unique()

for i, segment in enumerate(segments):
    data = top_countries_per_segment[top_countries_per_segment['Segment'] == segment]
    bars = axes[i].barh(data['Country'], data['Customer Count'], color='#4C72B0')
    axes[i].barh(data['Country'], data['Customer Count'], color='#4C72B0')
    axes[i].set_title(f'{segment}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Customers', fontsize=9)
    axes[i].invert_yaxis()

    # Add labels to each bar
    for bar in bars:
        axes[i].text(
            bar.get_width() + 1,
            bar.get_y() + bar.get_height() / 2,
            f'{int(bar.get_width())}',
            va='center',
            fontsize=12
        )

plt.suptitle('Top 5 Countries per Customer Segment', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()